# DIE04: MySQL Integration with SQLAlchemy

En entornos corporativos, los datos rara vez están en archivos CSV; generalmente residen en bases de datos relacionales (RDBMS) como MySQL, PostgreSQL o SQL Server. 

Para interactuar con estas bases de datos de forma segura y eficiente desde Python, utilizamos **SQLAlchemy** como nuestro motor de conexión (ORM) y `pymysql` como el controlador específico para MySQL.

In [ ]:
# Instalación de dependencias necesarias (descomentar si es necesario)
# %pip install pandas sqlalchemy pymysql

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# 1. Configurar los parámetros de conexión
# Nota: En un entorno real, estas credenciales deben estar en variables de entorno (.env)
DB_NAME = 'biblioteca'
DB_USER = 'root'
DB_PASSWORD = ''
DB_HOST = 'localhost'
DB_PORT = '3306'

# 2. Crear la URL de conexión de forma segura
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

# 3. Inicializar el motor de SQLAlchemy
engine = create_engine(connection_url)
print("Motor SQLAlchemy configurado exitosamente.")

Motor SQLAlchemy configurado exitosamente.


## 1. Reading Data (SQL to DataFrame)
Podemos ejecutar consultas SQL complejas (incluyendo `JOINs` y agregaciones) y cargar el resultado directamente en un DataFrame de Pandas usando `pd.read_sql_query()`. Esto delega el procesamiento pesado al servidor de base de datos.

In [4]:
print("--- Libros y sus Autores (INNER JOIN) ---")
query_libros = """
SELECT l.titulo, a.nombre AS autor
FROM libros l
INNER JOIN autores a
ON l.id_autor = a.id_autor;
"""
df_libros = pd.read_sql_query(query_libros, con=engine)
display(df_libros.head())

print("\n--- Historial de Préstamos con Múltiples JOINs ---")
query_prestamos = """
SELECT u.nombres, l.titulo, p.fecha_prestamo, p.estado
FROM prestamos p
JOIN usuarios u ON p.id_usuario = u.id_usuario
JOIN libros l ON p.id_libro = l.id_libro;
"""
df_prestamos = pd.read_sql_query(query_prestamos, con=engine)
display(df_prestamos.head())

print("\n--- Agregación SQL: Préstamos por Usuario ---")
query_agregacion = """
SELECT u.nombres, COUNT(*) AS total_prestamos
FROM prestamos p
JOIN usuarios u ON p.id_usuario = u.id_usuario
GROUP BY u.nombres;
"""
df_agregacion = pd.read_sql_query(query_agregacion, con=engine)
display(df_agregacion)

--- Libros y sus Autores (INNER JOIN) ---


,titulo,autor
0,Cien años de soledad,Gabriel García Márquez
1,El amor en los tiempos del cólera,Gabriel García Márquez
2,La ciudad y los perros,Mario Vargas Llosa
3,La casa de los espíritus,Isabel Allende
4,Viaje al centro de la Tierra,Julio Verne



--- Historial de Préstamos con Múltiples JOINs ---


,nombres,titulo,fecha_prestamo,estado
0,Ana Torres,Cien años de soledad,2025-01-10,Devuelto
1,Luis Pérez,La ciudad y los perros,2025-02-05,Devuelto
2,María López,1984,2025-03-01,Prestado
3,Ana Torres,Viaje al centro de la Tierra,2025-03-12,Devuelto
4,Carlos Ruiz,El amor en los tiempos del cólera,2025-04-08,Prestado



--- Agregación SQL: Préstamos por Usuario ---


,nombres,total_prestamos
0,Ana Torres,2
1,Luis Pérez,1
2,María López,1
3,Carlos Ruiz,1
4,José Ramírez,1


## 2. Writing Data (DataFrame to SQL)
El método `.to_sql()` permite insertar registros masivos desde un DataFrame hacia una tabla SQL. Usamos `if_exists="append"` para añadir registros sin sobreescribir la tabla existente, e `index=False` para no enviar el índice numérico de Pandas.

In [5]:
# Nuevos datos recolectados que necesitamos enviar a la base de datos
nuevos_autores = pd.DataFrame({
    "nombre": ["J. K. Rowling", "Stephen King", "Arthur Conan Doyle"],
    "nacionalidad": ["Reino Unido", "Estados Unidos", "Reino Unido"]
})

# Inserción en la base de datos
nuevos_autores.to_sql(
    name="autores",
    con=engine,
    if_exists="append",
    index=False
)

# Verificamos que se hayan insertado
df_autores_actualizados = pd.read_sql("SELECT * FROM autores;", engine)
print("--- Tabla de Autores Actualizada ---")
display(df_autores_actualizados.tail())

--- Tabla de Autores Actualizada ---


,id_autor,nombre,nacionalidad
3,4,Julio Verne,Francia
4,5,George Orwell,Reino Unido
5,6,J. K. Rowling,Reino Unido
6,7,Stephen King,Estados Unidos
7,8,Arthur Conan Doyle,Reino Unido


## 3. Executing Raw SQL Statements (Updates/Deletes)
Para operaciones que no devuelven datos (como `UPDATE`, `DELETE` o `INSERT` manual), abrimos una conexión transaccional con `engine.begin()` y usamos `text()` para enviar parámetros de forma segura, previniendo ataques de inyección SQL.

In [6]:
from sqlalchemy import text

# Actualizamos la nacionalidad de Stephen King usando un bloque de transacción
with engine.begin() as conn:
    conn.execute(
        text("""
            UPDATE autores
            SET nacionalidad = :nacionalidad
            WHERE nombre = :nombre
        """),
        {
            "nacionalidad": "EE.UU.",
            "nombre": "Stephen King"
        }
    )

# Verificamos el cambio
df_verificacion = pd.read_sql(
    "SELECT * FROM autores WHERE nombre = 'Stephen King';", 
    engine
)
print("--- Verificación de la Actualización ---")
display(df_verificacion)

--- Verificación de la Actualización ---


,id_autor,nombre,nacionalidad
0,7,Stephen King,EE.UU.
